In [ ]:
import pandas as pd
import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest

from sksurv.util import Surv
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from lifelines import CoxPHFitter

from lifelines.utils import concordance_index
from sklearn.inspection import permutation_importance

import xgboost as xgb
from sksurv.metrics import concordance_index_censored

### Importation datasets

In [ ]:
# inclusion = pl.read_parquet("Preprocessed_data/inclusion.parquet")
# image = pl.read_parquet("Preprocessed_data/image.parquet")
# creat = pl.read_parquet("Preprocessed_data/creat.parquet")
# genetics = pl.read_parquet("Preprocessed_data/genetics.parquet")
# center = pl.read_parquet("Preprocessed_data/center.parquet")
# last_follow_up = pl.read_parquet("Preprocessed_data/last_follow_up.parquet")

#### Préparation de la base de données

**Creat + inclusion**

In [ ]:
# # ==========================================================
# # PRÉPARATION DE LA BASE LONGITUDINALE
# # ==========================================================

# # Suppression des patients sans sexe renseigné

# inclusion_clean = inclusion.filter(
#     pl.col("sex").is_not_null()
# )


# # ==========================================================
# # JOINTURE DES DONNÉES BIOLOGIQUES ET CLINIQUES
# # ==========================================================

# df_long = (
#     creat.select([
#         "patient_id",
#         "creat_date",
#         "age_creat",
#         "ckd_epi"
#     ])
#     .join(
#         inclusion_clean.select([
#             "patient_id",
#             "center_id",
#             "birth_date",
#             "inclusion_date",
#             "sex",
#             "has_hypertension",
#             "age_hypertension",
#             "has_hypertension_before_35",
#             "has_urologic_event",
#             "age_cysts_pain",
#             "age_cysts_infection",
#             "age_intra_cysts_hemorrhage",
#             "age_lithiasis",
#             "age_macro_hematuria",
#             "has_urologic_event_before_35",
#         ]),
#         on="patient_id",
#         how="left"
#     )
#     .sort([
#         "patient_id",
#         "creat_date"
#     ])
# )
# print(
#     "Patients supprimés :",
#     inclusion.height - inclusion_clean.height
# )

# df_long.head()

In [ ]:
# # ==========================================================
# # AJOUT ET RECODAGE DES DONNÉES GÉNÉTIQUES
# # ==========================================================

# genetics_clean = (
#     genetics
#     .with_columns([

#         # --------------------------------------------------
#         # Type de variant
#         # --------------------------------------------------

#         pl.when(pl.col("status").str.to_uppercase() == "GUR")
#             .then(pl.lit("GUR"))

#         .when(
#             pl.col("status").is_null() |
#             pl.col("variant").is_null() |
#             (pl.col("variant").str.strip_chars() == "")
#         )
#             .then(pl.lit("inconnu"))

#         .when(
#             pl.col("variant")
#             .str.to_lowercase()
#             .is_in(["tru"])
#         )
#             .then(pl.lit("truncating"))

#         .when(
#             pl.col("variant").str.contains(
#                 r"(?i)del|dup|ins|fs|\*|splice|\+\d+|\-\d+"
#             )
#         )
#             .then(pl.lit("truncating"))

#         .otherwise(pl.lit("missense"))
#         .alias("variant_type"),

#         # --------------------------------------------------
#         # Catégorie génétique utilisée pour les modèles
#         # --------------------------------------------------

#         pl.when(pl.col("status") != "solved")
#         .then(pl.lit("GUR"))

#         .when(pl.col("gene_subgroup") == "PKD2")
#         .then(pl.lit("PKD2"))

#         .when(
#             (pl.col("gene_subgroup") == "PKD1") &
#             (pl.col("is_truncating") == True)
#         )
#         .then(pl.lit("PKD1_truncating"))

#         .when(pl.col("gene_subgroup") == "PKD1")
#         .then(pl.lit("PKD1_non_truncating"))

#         .otherwise(pl.lit("other"))
#         .alias("genetic")

#     ])
# )

# df_long = df_long.with_columns(
#     pl.col("patient_id")
#       .str.strip_chars()
#       .str.to_uppercase()
# )

# genetics_clean = genetics_clean.with_columns(
#     pl.col("patient_id")
#       .str.strip_chars()
#       .str.to_uppercase()
# )

# # ==========================================================
# # AJOUT DES DONNÉES GÉNÉTIQUES À LA BASE LONGITUDINALE
# # ==========================================================

# df_long_genetics = (
#     df_long
#     .join(
#         genetics_clean.select([
#             "patient_id",
#             "variant",
#             "variant_type",
#             "genetic"
#         ]),
#         on="patient_id",
#         how="left"
#     )
# )

# # ==========================================================
# # CONSERVATION DES PATIENTS AVEC UNE INFORMATION GÉNÉTIQUE
# # ==========================================================

# df_long_genetics = (
#     df_long_genetics
#     .filter(pl.col("genetic").is_not_null())
# )

# df_long_genetics.head()

In [ ]:
# # ==========================================================
# # AJOUT DES DONNÉES DE SUIVI
# # ==========================================================

# df_survival_long = (
#     df_long_genetics
#     .join(
#         last_follow_up.select([
#             "patient_id",
#             "dialysis_date",
#             "transplant_date",
#             "kidney_failure_date",
#             "last_follow_up_date",
#             "is_alive",
#             "death_date",
#             "last_creat_date",
#             "is_on_renal_replacement"
#         ]),
#         on="patient_id",
#         how="left"
#     )
# )


# # ==========================================================
# # DÉFINITION DE LA DATE D'ÉVÉNEMENT
# # ==========================================================

# df_survival_long = df_survival_long.with_columns(

#     pl.min_horizontal([
#         pl.col("kidney_failure_date"),
#         pl.col("dialysis_date"),
#         pl.col("transplant_date")
#     ]).alias("event_date")

# )


# # ==========================================================
# # ÉVÉNEMENT IRT
# # ==========================================================

# df_survival_long = df_survival_long.with_columns(

#     pl.col("event_date")
#     .is_not_null()
#     .cast(pl.Int8)
#     .alias("event_irt")

# )


# # ==========================================================
# # SUPPRESSION DES MESURES APRÈS IRT
# # ==========================================================

# df_survival_long = df_survival_long.filter(

#     (pl.col("event_date").is_null()) |
#     (pl.col("creat_date") <= pl.col("event_date"))

# )

In [ ]:
# # ==========================================================
# # ÂGE DE PREMIÈRE COMPLICATION UROLOGIQUE
# # ==========================================================

# df_survival_long = df_survival_long.with_columns(

#     pl.min_horizontal([
#         "age_cysts_pain",
#         "age_cysts_infection",
#         "age_intra_cysts_hemorrhage",
#         "age_lithiasis",
#         "age_macro_hematuria"
#     ]).alias("age_urologic_event")

# )


# # ==========================================================
# # GESTION DU SUIVI ET CALCUL DU TEMPS AVANT ÉVÉNEMENT
# # ==========================================================

# df_survival_long = (
#     df_survival_long

#     # --------------------------------------------------
#     # Nettoyage des incohérences après décès
#     # --------------------------------------------------

#     .with_columns([

#         pl.when(
#             (pl.col("death_date").is_not_null()) &
#             (pl.col("last_follow_up_date") > pl.col("death_date"))
#         )
#         .then(None)
#         .otherwise(pl.col("last_follow_up_date"))
#         .alias("last_follow_up_date_clean"),

#         pl.when(
#             (pl.col("death_date").is_not_null()) &
#             (pl.col("last_creat_date") > pl.col("death_date"))
#         )
#         .then(None)
#         .otherwise(pl.col("last_creat_date"))
#         .alias("last_creat_date_clean")

#     ])

#     # --------------------------------------------------
#     # Définition de la date finale de suivi
#     # --------------------------------------------------

#     .with_columns(

#         pl.when(pl.col("event_irt") == 1)
#         .then(pl.col("event_date"))

#         .when(pl.col("death_date").is_not_null())
#         .then(pl.col("death_date"))

#         .otherwise(
#             pl.coalesce([
#                 pl.col("last_follow_up_date_clean"),
#                 pl.col("last_creat_date_clean")
#             ])
#         )

#         .alias("end_date")

#     )

#     # --------------------------------------------------
#     # Temps restant avant événement ou censure
#     # --------------------------------------------------

#     .with_columns(

#         (
#             (pl.col("end_date") - pl.col("creat_date"))
#             .dt.total_days() / 365.25
#         )
#         .alias("time_to_event")

#     )
# )

# print("La taille du dataset :", df_survival_long.shape)

# print(
#     "Nombre de patients :",
#     df_survival_long.select(
#         pl.col("patient_id").n_unique()
#     )
# )

# df_survival_long.head(10)

#### Nettoyage de la base de données

In [ ]:
# # ==========================================================
# # NETTOYAGE DE LA COHORTE LONGITUDINALE
# # ==========================================================

# # --------------------------------------------------
# # Effectifs initiaux
# # --------------------------------------------------

# n_patients_initial = (
#     df_survival_long
#     .select(pl.col("patient_id").n_unique())
#     .item()
# )

# n_obs_initial = df_survival_long.height


# # --------------------------------------------------
# # Âge à la première mesure biologique
# # --------------------------------------------------

# first_creat_age = (
#     df_survival_long
#     .group_by("patient_id")
#     .agg(
#         pl.col("age_creat")
#         .min()
#         .alias("first_creat_age")
#     )
# )

# eligible_patients = (
#     first_creat_age
#     .filter(pl.col("first_creat_age") >= 16)
#     .select("patient_id")
# )

# df_adult = (
#     df_survival_long
#     .join(
#         eligible_patients,
#         on="patient_id",
#         how="inner"
#     )
# )


# # --------------------------------------------------
# # Données biologiques disponibles
# # --------------------------------------------------

# df_biology = (
#     df_adult
#     .filter(
#         pl.col("creat_date").is_not_null()
#         & pl.col("ckd_epi").is_not_null()
#     )
# )


# # --------------------------------------------------
# # Prédiction avant le stade CKD4 avancé
# # --------------------------------------------------

# df_ckd = (
#     df_biology
#     .filter(
#         pl.col("ckd_epi") >= 30
#     )
# )


# # --------------------------------------------------
# # Contrôle de cohérence temporelle
# # --------------------------------------------------

# df_survival_clean = (
#     df_ckd
#     .filter(
#         (pl.col("creat_date") >= pl.col("birth_date"))
#         & (pl.col("end_date") >= pl.col("creat_date"))
#         & (pl.col("time_to_event") >= 0)
#         & ~(
#             (pl.col("event_irt") == 0)
#             & (pl.col("time_to_event") == 0)
#         )
#     )
#     .sort([
#         "patient_id",
#         "creat_date"
#     ])
# )


# # ==========================================================
# # RÉSUMÉ DE LA SÉLECTION DE LA COHORTE
# # ==========================================================

# summary = [
#     ("Cohorte initiale", df_survival_long),
#     ("Patients ≥16 ans", df_adult),
#     ("Données biologiques disponibles", df_biology),
#     ("DFG ≥30 mL/min/1.73m²", df_ckd),
#     ("Cohorte finale", df_survival_clean),
# ]

# print("\n================ Sélection de la cohorte ================\n")

# for label, df in summary:

#     n_patients = (
#         df
#         .select(pl.col("patient_id").n_unique())
#         .item()
#     )

#     print(
#         f"{label:<35}"
#         f"{n_patients:>5} patients | "
#         f"{df.height:>6} observations"
#     )

In [ ]:
# # ==========================================================
# # DESCRIPTION DE LA COHORTE LONGITUDINALE
# # ==========================================================

# # -----------------------------
# # Statistiques au niveau patient
# # -----------------------------

# patient_summary = (
#     df_survival_clean
#     .group_by("patient_id")
#     .agg(
#         pl.len().alias("n_measures"),
#         pl.col("time_to_event").max().alias("follow_up"),
#         pl.col("event_irt").max().alias("event_irt")
#     )
# )

# # -----------------------------
# # Effectifs
# # -----------------------------

# n_rows = df_survival_clean.height
# n_patients = patient_summary.height

# # -----------------------------
# # Mesures répétées
# # -----------------------------

# measures_stats = (
#     patient_summary
#     .select(
#         pl.col("n_measures").median().alias("median"),
#         pl.col("n_measures").min().alias("min"),
#         pl.col("n_measures").max().alias("max")
#     )
# )

# median_measures = measures_stats["median"][0]
# min_measures = measures_stats["min"][0]
# max_measures = measures_stats["max"][0]

# # -----------------------------
# # Suivi et événements
# # -----------------------------

# median_follow_up = (
#     patient_summary
#     .select(
#         pl.col("follow_up").median()
#     )
#     .item()
# )

# irt_rate = (
#     patient_summary
#     .select(
#         pl.col("event_irt").mean()
#     )
#     .item()
# )

# # ==========================================================
# # RÉSUMÉ DE LA COHORTE
# # ==========================================================

# print("\n================ Description de la cohorte ================\n")

# print(f"Nombre total d'observations : {n_rows}")
# print(f"Nombre de patients          : {n_patients}")

# print("\nMesures par patient")
# print(f"  Médiane : {median_measures}")
# print(f"  Minimum : {min_measures}")
# print(f"  Maximum : {max_measures}")

# print("\nSuivi")
# print(f"  Temps médian de suivi : {median_follow_up:.2f} ans")

# print("\nÉvénement IRT")
# print(f"  Patients avec IRT : {irt_rate:.2%}")

In [ ]:
# # ==========================================================
# # BASE DE DÉPART POUR LA CONSTRUCTION DU DATASET DYNAMIQUE
# # ==========================================================

# # La cohorte nettoyée constitue la base utilisée pour générer
# # les observations longitudinales de prédiction.

# df_survival_dynamic = df_survival_clean

# print(
#     f"Base dynamique : "
#     f"{df_survival_dynamic.select(pl.col('patient_id').n_unique()).item()} patients | "
#     f"{df_survival_dynamic.height} observations"
# )

In [ ]:
# # ==========================================================
# # CONSTRUCTION DES VARIABLES CLINIQUES DYNAMIQUES
# # ==========================================================

# def build_dynamic_clinical(age_event_col: str,
#                            has_event_col: str,
#                            status_name: str,
#                            age_name: str):
#     """
#     Construit deux variables dépendantes du temps :

#     status :
#         0 = événement non survenu à la date de la visite
#         1 = événement déjà survenu (âge connu)
#         2 = événement présent mais âge inconnu

#     age :
#         âge de survenue de l'événement uniquement
#         lorsqu'il est déjà survenu, sinon null.
#     """

#     status = (

#         pl.when(
#             pl.col(age_event_col).is_not_null()
#             &
#             (pl.col("age_creat") >= pl.col(age_event_col))
#         )
#         .then(pl.lit(1))

#         .when(
#             (pl.col(has_event_col) == 1)
#             &
#             pl.col(age_event_col).is_null()
#         )
#         .then(pl.lit(2))

#         .otherwise(pl.lit(0))
#         .alias(status_name)

#     )

#     age = (

#         pl.when(
#             pl.col(age_event_col).is_not_null()
#             &
#             (pl.col("age_creat") >= pl.col(age_event_col))
#         )
#         .then(pl.col(age_event_col))

#         .otherwise(None)
#         .alias(age_name)

#     )

#     return [status, age]

In [ ]:
# df_survival_dynamic = (

#     df_survival_dynamic

#     .with_columns(

#         build_dynamic_clinical(
#             age_event_col="age_hypertension",
#             has_event_col="has_hypertension",
#             status_name="hypertension_at_t",
#             age_name="age_hypertension_at_t"
#         )

#         +

#         build_dynamic_clinical(
#             age_event_col="age_urologic_event",
#             has_event_col="has_urologic_event",
#             status_name="urologic_event_at_t",
#             age_name="age_urologic_event_at_t"
#         )

#     )

# )

In [ ]:
# # ==========================================================
# # APERÇU DES VARIABLES DYNAMIQUES
# # ==========================================================

# df_survival_dynamic.select([

#     "patient_id",
#     "creat_date",
#     "age_creat",
#     "age_hypertension",

#     "has_hypertension",
#     "hypertension_at_t",
#     "age_hypertension_at_t",

#     "has_urologic_event",
#     "urologic_event_at_t",
#     "age_urologic_event_at_t",

#     "end_date",

#     "time_to_event",
#     "event_irt"

# ]).head(10)

#### Séparation de la base de données

In [ ]:
# # ==========================================================
# # VALIDATION EXTERNE PAR CENTRE
# # ==========================================================

# # Centres réservés à la validation externe.
# # Ils sont exclus de la phase de développement afin d'évaluer
# # la capacité de généralisation du modèle sur des centres indépendants.

# external_centers = ["09", "04"]

# # Cohorte de développement
# df_development = (
#     df_survival_dynamic
#     .filter(
#         ~pl.col("center_id").is_in(external_centers)
#     )
# )

# # Cohorte de validation externe
# df_external_test = (
#     df_survival_dynamic
#     .filter(
#         pl.col("center_id").is_in(external_centers)
#     )
# )


# # ==========================================================
# # RÉSUMÉ DES COHORTES
# # ==========================================================

# def summarize_cohort(df, cohort_name):
#     """Affiche les caractéristiques principales d'une cohorte."""

#     patient_summary = (
#         df
#         .group_by("patient_id")
#         .agg(
#             pl.col("event_irt").max().alias("event_irt")
#         )
#     )

#     n_patients = patient_summary.height
#     n_events = (
#         patient_summary
#         .select(pl.col("event_irt").sum())
#         .item()
#     )

#     event_rate = (
#         patient_summary
#         .select(pl.col("event_irt").mean())
#         .item()
#     )

#     print(
#         f"{cohort_name:<22}"
#         f"{n_patients:>5} patients | "
#         f"{df.height:>6} observations | "
#         f"{n_events:>4} événements ({event_rate:.1%})"
#     )


# print("\n================ Validation externe ================\n")

# summarize_cohort(df_development, "Développement")

# summarize_cohort(df_external_test, "Validation externe")

In [ ]:
# ==========================================================
# SAUVEGARDE DES COHORTES LONGITUDINALES
# ==========================================================

df_development.write_parquet(
    "My_data/adpkd_longitudinal_development.parquet"
)

df_external_test.write_parquet(
    "My_data/adpkd_longitudinal_external_test.parquet"
)

#### Split train test

In [ ]:
# ==================================================
# IMPORTATION DU JEU DE DONNEES
# ==================================================

df_development = pl.read_parquet("My_data/adpkd_longitudinal_development.parquet")
#df_external_test = pl.read_parquet("My_data/adpkd_longitudinal_external_test.parquet")

# ==========================================================
# REPRODUCTIBILITÉ
# ==========================================================

SEED = 42


# ==========================================================
# SPLIT TRAIN / TEST (PATIENT-LEVEL)
# ==========================================================

# Patients uniques avec statut IRT

patients = (
    df_development
    .select(["patient_id", "event_irt"])
    .group_by("patient_id")
    .agg(
        pl.max("event_irt").alias("event_irt")
    )
    .sort("patient_id") 
    .to_pandas()
)

# Split stratifié 80 / 20
train_ids, test_ids = train_test_split(
    patients,
    test_size=0.2,
    stratify=patients["event_irt"],
    random_state=SEED
)

# Cohorte train

df_train = df_development.filter(
    pl.col("patient_id").is_in(train_ids["patient_id"])
)

# Cohorte test 

df_test = df_development.filter(
    pl.col("patient_id").is_in(test_ids["patient_id"])
)

# ==========================================================
# RÉSUMÉ D'UNE COHORTE
# ==========================================================

def summarize_split(df, name):
    """
    Affiche les caractéristiques principales d'une cohorte.
    """

    patient_summary = (
        df
        .group_by("patient_id")
        .agg(
            pl.max("event_irt").alias("event_irt")
        )
    )

    n_patients = patient_summary.height
    n_events = (
        patient_summary
        .filter(pl.col("event_irt") == 1)
        .height
    )

    event_rate = n_events / n_patients
    n_observations = df.height

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print(f"Patients       : {n_patients}")
    print(f"Observations   : {n_observations}")
    print(f"Événements IRT : {n_events}")
    print(f"Taux IRT       : {event_rate:.2%}")

In [ ]:
# ==========================================================
# RÉSUMÉ DES COHORTES
# ==========================================================

summarize_split(df_train, "TRAIN")

summarize_split(df_test, "TEST")

#### CONSTRUCTION DU DATASET DYNAMIQUE 

In [ ]:
def build_dynamic_dataset(df):
    """
    Construit un dataset longitudinal dynamique.

    Chaque ligne du dataset correspond à une visite et est
    construite uniquement à partir des informations
    disponibles jusqu'à cette visite.
    """

    snapshots = []

    for patient_id, patient_df in df.group_by("patient_id"):

        # Tri chronologique des visites
        patient_df = patient_df.sort("creat_date")

        n_obs = patient_df.height

        # Construction d'un snapshot à chaque visite
        for i in range(n_obs):

            current_visit = patient_df.row(i, named=True)

            # --------------------------------------------------
            # Nombre cumulé de mesures disponibles
            # au moment de la visite courante
            # --------------------------------------------------

            n_measures = i + 1

            # --------------------------------------------------
            # Historique DFG disponible
            # --------------------------------------------------

            dfg_history = (
                patient_df["ckd_epi"][:n_measures]
                .to_list()
            )

            # Tableau NumPy utilisé pour les calculs numériques
            egfr = np.asarray(
                dfg_history,
                dtype=float
            )

            # --------------------------------------------------
            # Historique temporel (années depuis la première mesure)
            # --------------------------------------------------

            time_history = (
                (
                    patient_df["creat_date"][:n_measures]
                    - patient_df["creat_date"][0]
                )
                .dt.total_days()
                .to_numpy()
                / 365.25
            )

            # Liste conservée pour les modèles séquentiels
            time_history_years = (
                time_history
                .tolist()
            )

            # --------------------------------------------------
            # Historique des âges (temps absolu)
            # --------------------------------------------------

            age_history = (
                patient_df["age_creat"][:n_measures]
                .to_list()
            )

            # --------------------------------------------------
            # Historique HTA
            # --------------------------------------------------

            raw_hypertension_history  = (
                patient_df["hypertension_at_t"][:n_measures]
                .to_list()
            )

            hypertension_age_history = (
                patient_df["age_hypertension_at_t"][:n_measures]
                .to_list()
            )

            # Présence d'une hypertension (0 = non, 1 = oui)

            hypertension_history   = [
                int(x > 0)
                for x in raw_hypertension_history
            ]

            # Date de survenue connue
            # 1 = date connue
            # 0 = absence d'HTA ou date inconnue

            hypertension_date_known_history  = [
                int(x == 1)
                for x in raw_hypertension_history
            ]

            # Nombre d'années depuis l'apparition de l'HTA

            years_since_hypertension_history = []

            for age_visit, age_ht in zip(age_history, hypertension_age_history):

                if age_ht is None:
                   years_since_hypertension_history.append(0.0)

                else:
                    years_since_hypertension_history.append(
                        float(age_visit - age_ht)
                    )

            # --------------------------------------------------
            # Historique des complications urologiques
            # --------------------------------------------------

            raw_urologic_history = (
                patient_df["urologic_event_at_t"][:n_measures]
                .to_list()
            )

            urologic_age_history = (
                patient_df["age_urologic_event_at_t"][:n_measures]
                .to_list()
            )
            # Présence d'une hypertension (0 = non, 1 = oui)

            urologic_history = [
                int(x > 0)
                for x in raw_urologic_history
            ]

            # Date de survenue connue
            # 1 = date connue
            # 0 = absence d'HTA ou date inconnue

            urologic_date_known_history   = [
                int(x == 1)
                for x in raw_urologic_history
            ]

            # Nombre d'années depuis la première complication

            years_since_urologic_event_history = []

            for age_visit, age_uro in zip(age_history, urologic_age_history):

                if age_uro is None:
                   years_since_urologic_event_history.append(0.0)

                else:
                     years_since_urologic_event_history.append(
                         float(age_visit - age_uro)
                     )

            
            # --------------------------------------------------
            # Temps écoulé depuis la première mesure
            # --------------------------------------------------

            time_since_first_measure = (
                time_history[-1]
            )

            # --------------------------------------------------
            # Conditions de calcul des pentes
            # --------------------------------------------------

            can_compute_global_slope = (
                (n_measures >= 3)
                and (time_since_first_measure >= 0.25)
            )

            can_compute_recent_slope = (
                (n_measures >= 3)
                and (
                    time_history[-1]
                    - time_history[-3]
                    >= 0.5
                )
            )

            # --------------------------------------------------
            # Pente cumulée du DFG
            # --------------------------------------------------

            if can_compute_global_slope:

                egfr_slope = float(
                    np.polyfit(
                        time_history,
                        egfr,
                        deg=1
                    )[0]
                )

            else:

                egfr_slope = np.nan

            # --------------------------------------------------
            # Pente récente (3 dernières mesures)
            # --------------------------------------------------

            if can_compute_recent_slope:

                recent_egfr_slope = float(
                    np.polyfit(
                        time_history[-3:],
                        egfr[-3:],
                        deg=1
                    )[0]
                )

            else:

                recent_egfr_slope = np.nan

            # --------------------------------------------------
            # DFG de référence
            # --------------------------------------------------

            baseline_ckd_epi = (
                egfr[0]
            )

            # --------------------------------------------------
            # DFG précédent
            # --------------------------------------------------

            previous_ckd_epi = (
                current_visit["ckd_epi"]
                if n_measures == 1
                else egfr[-2]
            )
            
            # --------------------------------------------------
            # Statistiques du DFG
            # --------------------------------------------------

            egfr_mean = float(
                egfr.mean()
            )

            egfr_std = (
                float(
                    egfr.std(ddof=1)
                )
                if n_measures >= 2
                else np.nan
            )

            egfr_min = float(
                egfr.min()
            )

            egfr_max = float(
                egfr.max()
            )

            egfr_cv = (
                egfr_std / egfr_mean
                if (
                    n_measures >= 2
                    and egfr_mean > 0
                )
                else np.nan
            )

            egfr_iqr = (
                float(
                    np.percentile(egfr, 75)
                    - np.percentile(egfr, 25)
                )
                if n_measures >= 4
                else np.nan
            )

            # --------------------------------------------------
            # Variation depuis la baseline
            # --------------------------------------------------

            delta_ckd_from_baseline = (
                current_visit["ckd_epi"]
                - baseline_ckd_epi
            )

            percent_decline_from_baseline = (
                100
                * delta_ckd_from_baseline
                / baseline_ckd_epi
                if baseline_ckd_epi > 0
                else np.nan
            )

            # --------------------------------------------------
            # Variation récente
            # --------------------------------------------------

            recent_delta_ckd = (
                egfr[-1] - egfr[-2]
                if n_measures >= 2
                else np.nan
            )

            # --------------------------------------------------
            # Intervalles entre mesures
            # --------------------------------------------------

            measurement_intervals = np.diff(time_history)

            # --------------------------------------------------
            # Statistiques des intervalles
            # --------------------------------------------------

            if measurement_intervals.size == 0:

                time_since_last_measure = np.nan
                mean_measurement_interval = np.nan
                std_measurement_interval = np.nan

            else:

                time_since_last_measure = float(
                    measurement_intervals[-1]
                )

                mean_measurement_interval = float(
                    measurement_intervals.mean()
                )

                std_measurement_interval = (
                    float(
                        measurement_intervals.std(ddof=1)
                    )
                    if measurement_intervals.size >= 2
                    else np.nan
                )


            # --------------------------------------------------
            # Variation annuelle récente
            # --------------------------------------------------

            if (
                measurement_intervals.size > 0
                and measurement_intervals[-1] > 0
            ):

                recent_annualized_change = (
                    recent_delta_ckd
                    / measurement_intervals[-1]
                )

            else:

                recent_annualized_change = np.nan

            #------------------------------------------------
            # Liste conservée pour les modèles séquentiels
            # -----------------------------------------------
            
            measurement_intervals = (
                measurement_intervals.tolist()
            )

            # --------------------------------------------------
            # Densité du suivi
            # --------------------------------------------------

            followup_density = (
                (n_measures - 1)
                / time_since_first_measure
                if time_since_first_measure >= 0.25
                else np.nan
            )

            # --------------------------------------------------
            # Résumé clinique
            # --------------------------------------------------

            ever_hypertension = int(
                current_visit["hypertension_at_t"] > 0
            )

            ever_urologic_event = int(
                current_visit["urologic_event_at_t"] > 0
            )

            # --------------------------------------------------
            # Temps depuis l'hypertension
            # --------------------------------------------------

            if (
                current_visit["age_hypertension_at_t"] is not None
            ):

                years_since_hypertension = (
                    current_visit["age_creat"]
                    - current_visit["age_hypertension_at_t"]
                )

            else:

                years_since_hypertension = np.nan

            # --------------------------------------------------
            # Temps depuis la première complication urologique
            # --------------------------------------------------

            if (
                current_visit["age_urologic_event_at_t"] is not None
            ):

                years_since_urologic_event = (
                    current_visit["age_creat"]
                    - current_visit["age_urologic_event_at_t"]
                )

            else:

                years_since_urologic_event = np.nan

            # --------------------------------------------------
            # Position dans le suivi
            # --------------------------------------------------

            is_first_measure = int(
                n_measures == 1
            )

            is_second_measure = int(
                n_measures == 2
            )

            has_longitudinal_history = int(
                n_measures >= 3
            )

            # --------------------------------------------------
            # Âge à la première mesure
            # --------------------------------------------------

            baseline_age = (
                patient_df["age_creat"][0]
            )
            
            # --------------------------------------------------
            # Snapshot
            # --------------------------------------------------

            snapshots.append({

                # ==========================================
                # IDENTIFIANT
                # ==========================================

                "patient_id":
                    current_visit["patient_id"],

                # ==========================================
                # VARIABLES FIXES
                # ==========================================

                "sex":
                    current_visit["sex"],

                "genetic":
                    current_visit["genetic"],

                # ==========================================
                # VARIABLES CLINIQUES ACTUELLES
                # ==========================================

                "age_creat":
                    current_visit["age_creat"],

                "hypertension_at_t":
                    current_visit["hypertension_at_t"],

                "age_hypertension_at_t":
                    current_visit["age_hypertension_at_t"],

                "urologic_event_at_t":
                    current_visit["urologic_event_at_t"],

                "age_urologic_event_at_t":
                    current_visit["age_urologic_event_at_t"],

                # ==========================================
                # VARIABLES RÉNALES ACTUELLES
                # ==========================================

                "current_ckd_epi":
                    current_visit["ckd_epi"],

                # ==========================================
                # HISTORIQUES (ODE-RNN / CDE)
                # ==========================================

                "egfr_history":
                    dfg_history,

                "time_history":
                    time_history_years,

                "age_history":
                    age_history,

                "hypertension_history":
                    hypertension_history,

                "hypertension_age_history":
                    hypertension_age_history,

                "urologic_history":
                    urologic_history,

                "urologic_age_history":
                    urologic_age_history,

                "measurement_intervals":
                    measurement_intervals,

                # ==========================================
                # VARIABLES DYNAMIQUES
                # ==========================================

                "n_measures":
                    n_measures,

                "time_since_first_measure":
                    time_since_first_measure,

                "time_since_last_measure":
                    time_since_last_measure,

                "followup_density":
                    followup_density,

                # ==========================================
                # FEATURES LONGITUDINALES
                # ==========================================

                "baseline_ckd_epi":
                    baseline_ckd_epi,

                "previous_ckd_epi":
                    previous_ckd_epi,

                "egfr_slope":
                    egfr_slope,

                "recent_egfr_slope":
                    recent_egfr_slope,

                "recent_annualized_change":
                    recent_annualized_change,

                "egfr_mean":
                    egfr_mean,

                "egfr_std":
                    egfr_std,

                "egfr_min":
                    egfr_min,

                "egfr_max":
                    egfr_max,

                "egfr_cv":
                    egfr_cv,

                "egfr_iqr":
                    egfr_iqr,

                "delta_ckd_from_baseline":
                    delta_ckd_from_baseline,

                "percent_decline_from_baseline":
                    percent_decline_from_baseline,

                "recent_delta_ckd":
                    recent_delta_ckd,

                "mean_measurement_interval":
                    mean_measurement_interval,

                "std_measurement_interval":
                    std_measurement_interval,

                "baseline_age":
                    baseline_age,

                "ever_hypertension":
                    ever_hypertension,

                "ever_urologic_event":
                    ever_urologic_event,

                "years_since_hypertension":
                    years_since_hypertension,

                "years_since_urologic_event":
                    years_since_urologic_event,

                "hypertension_history":
                    hypertension_history,

                "hypertension_date_known_history":
                    hypertension_date_known_history,

                "years_since_hypertension_history":
                    years_since_hypertension_history,

                "urologic_history":
                    urologic_history,

                "urologic_date_known_history":
                    urologic_date_known_history,

                "years_since_urologic_event_history":
                    years_since_urologic_event_history,

                "is_first_measure":
                    is_first_measure,

                "is_second_measure":
                    is_second_measure,

                "has_longitudinal_history":
                    has_longitudinal_history,

                # ==========================================
                # INFORMATIONS DU SNAPSHOT
                # ==========================================

                "creat_date":
                    current_visit["creat_date"],

                # ==========================================
                # CIBLE
                # ==========================================

                "time_to_event":
                    current_visit["time_to_event"],

                "event_irt":
                    current_visit["event_irt"]

            })

    return pl.DataFrame(snapshots)

#### CONSTRUCTION DES DATASETS DYNAMIQUES

In [ ]:
# Jeu d'entraînement
train_dynamic = build_dynamic_dataset(
    df_train
)

# Jeu de test interne
internal_test_dynamic = build_dynamic_dataset(
    df_test
)

# Jeu de validation externe
# external_test_dynamic = build_dynamic_dataset(
#     df_external_test
# )

# ==========================================================
# RÉSUMÉ
# ==========================================================

print("\n=========== Datasets dynamiques ===========\n")

print(
    f"Train               : {train_dynamic.height} snapshots | "
    f"{train_dynamic.select(pl.col('patient_id').n_unique()).item()} patients"
)

print(
    f"Test interne        : {internal_test_dynamic.height} snapshots | "
    f"{internal_test_dynamic.select(pl.col('patient_id').n_unique()).item()} patients"
)

# print(
#     f"Validation externe  : {external_test_dynamic.height} snapshots | "
#     f"{external_test_dynamic.select(pl.col('patient_id').n_unique()).item()} patients"
# )

In [ ]:
# # ==========================================================
# # SAUVEGARDE DES COHORTES LONGITUDINALES
# # ==========================================================

train_dynamic.write_parquet(
    "My_data/train_dynamic.parquet"
)

internal_test_dynamic.write_parquet(
    "My_data/internal_test_dynamic.parquet"
)

# external_test_dynamic.write_parquet(
#     "My_data/external_test_dynamic.parquet"
# )

In [ ]:
# train_dynamic.glimpse()

In [ ]:
train_dynamic.select([
    "egfr_mean",
    "egfr_std",
    "egfr_slope",
    "followup_density",
    "years_since_hypertension",
    "years_since_urologic_event"
]).describe()

In [ ]:
df_train.filter(
    pl.col("patient_id") == "202213"
).select([
    "creat_date",
    "age_creat",
    "ckd_epi",
    "time_to_event"
]).sort("creat_date")

In [ ]:
train_dynamic.filter(
    pl.col("egfr_slope") < -20
).select([
    "patient_id",
    "n_measures",
    "time_since_first_measure",
    "current_ckd_epi",
    "baseline_ckd_epi",
    "egfr_slope"
])

##### Quelques vérifications

In [ ]:
(
    train_dynamic
    .filter(
        pl.col("patient_id") == train_dynamic["patient_id"][0]
    )
    .select([
        "patient_id",
        "n_measures",
        "age_creat",
        "creat_date",
        "current_ckd_epi",
        "egfr_history",
        "time_history",
        "egfr_slope",
        "hypertension_at_t",
        "urologic_event_at_t",
        "time_to_event",
        "event_irt"
    ])
)

In [ ]:
patient_df = (
    train_dynamic
    .filter(pl.col("patient_id") == "060015")
    .sort("n_measures")
)

patient_df.select([
    "n_measures",
    "current_ckd_epi",
    "egfr_slope",
    "time_to_event"
])

In [ ]:
train_dynamic.select(
    pl.col("time_since_first_measure")
).describe()

In [ ]:
train_dynamic.group_by(
    "n_measures"
).agg([
    pl.len().alias("n"),
    pl.col("event_irt").mean().alias("event_rate"),
    pl.col("time_to_event").median().alias("median_time_to_event")
]).sort("n_measures")

In [ ]:
(
    train_dynamic
    .group_by("n_measures")
    .agg([
        pl.len().alias("n"),
        pl.col("event_irt").mean().alias("event_rate")
    ])
    .sort("n_measures")
).head(12)

In [ ]:
# ==========================================================
# CONSTRUCTION DU DATASET À k VISITES
# ==========================================================

def build_k_visit_dataset(
    df_dynamic,
    k
):

    return (

        df_dynamic

        # Conservation des snapshots disponibles
        .filter(
            pl.col("n_measures") <= k
        )
        .filter(
            pl.col("time_to_event") > 0
        )

        .sort([
            "patient_id",
            "n_measures"
        ])

        .group_by("patient_id")
        .last()

        .sort("patient_id")

    )


# ==========================================================
# CONSTRUCTION DES DATASETS 1 → 12 MESURES
# ==========================================================

train_datasets = {}
internal_test_datasets = {}
# external_test_datasets = {}

for k in range(1, 13):

    train_datasets[k] = build_k_visit_dataset(
        train_dynamic,
        k=k
    )

    internal_test_datasets[k] = build_k_visit_dataset(
        internal_test_dynamic,
        k=k
    )

    # external_test_datasets[k] = build_k_visit_dataset(
    #     external_test_dynamic,
    #     k=k
    # )

    print(
        f"k = {k:2d} | "
        f"Train : {train_datasets[k].height:4d} | "
        f"Test interne : {internal_test_datasets[k].height:4d} | "
        # f"Test externe : {datasets_external_test[k].height:4d}"
    )

In [ ]:
for k in range(1, 13):

    train_data = train_datasets[k]

    test_data = internal_test_datasets[k]


### MODELISATION

In [ ]:
# ==========================================================
# VARIABLES DU MODÈLE
# ==========================================================

FEATURES = [

    # --------------------------
    # Démographie
    # --------------------------

    "sex",
    "genetic",

    # --------------------------
    # Visite actuelle
    # --------------------------

    "age_creat",
    "current_ckd_epi",

    # --------------------------
    # Variables cliniques dynamiques
    # --------------------------

    "hypertension_at_t",
    "age_hypertension_at_t",

    "urologic_event_at_t",
    "age_urologic_event_at_t",

    # --------------------------
    # Historique DFG
    # --------------------------

    "baseline_ckd_epi",
    "previous_ckd_epi",
    "delta_ckd_from_baseline",

    "recent_annualized_change",

    "egfr_mean",
    "egfr_std",
    "egfr_slope",

    # --------------------------
    # Suivi
    # --------------------------

    "time_since_last_measure",
    "followup_density",
    "n_measures"

]

#### FONCTIONS DE LA PREPARATION DES DONNEES

In [ ]:
# ==========================================================
# PRÉPARATION DES DONNÉES
# ==========================================================

def prepare_dataset(
    train_df,
    test_df,
    features
):
    """
    Prépare les jeux d'entraînement et de test
    pour XGBoost Survival AFT.

    Parameters
    ----------
    train_df : pl.DataFrame
    test_df : pl.DataFrame
    features : list[str]

    Returns
    -------
    X_train : pandas.DataFrame
    X_test : pandas.DataFrame
    y_train : pandas.DataFrame
    y_test : pandas.DataFrame
    test_info : pandas.DataFrame
    """

    # --------------------------------------------------
    # Conversion Polars -> Pandas
    # --------------------------------------------------

    train_pd = train_df.to_pandas()

    test_pd = test_df.to_pandas()

    # --------------------------------------------------
    # Vérification des variables
    # --------------------------------------------------

    missing_train = (
        set(features)
        - set(train_pd.columns)
    )

    missing_test = (
        set(features)
        - set(test_pd.columns)
    )

    assert len(missing_train) == 0, (
        f"Variables manquantes dans train : {missing_train}"
    )

    assert len(missing_test) == 0, (
        f"Variables manquantes dans test : {missing_test}"
    )

    # --------------------------------------------------
    # Colonnes de survie
    # --------------------------------------------------

    target_columns = [

        "patient_id",

        "time_to_event",

        "event_irt"

    ]

    # --------------------------------------------------
    # Variables explicatives
    # --------------------------------------------------

    X_train = train_pd[
        features
    ].copy()

    X_test = test_pd[
        features
    ].copy()

    categorical_features = [
        "sex",
        "genetic",
        "hypertension_at_t",
        "urologic_event_at_t",
    ]

    for col in categorical_features:

        X_train[col] = X_train[col].astype("category")
        X_test[col] = X_test[col].astype("category")

    # --------------------------------------------------
    # Variables de survie
    # --------------------------------------------------

    y_train = train_pd[
        target_columns
    ].copy()

    y_test = test_pd[
        target_columns
    ].copy()

    # --------------------------------------------------
    # Informations nécessaires à l'évaluation
    # --------------------------------------------------

    test_info = test_pd[

        target_columns
        + ["n_measures"]

    ].copy()

    return (

        X_train,

        X_test,

        y_train,

        y_test,

        test_info

    )

In [ ]:
# ==========================================================
# LABELS POUR XGBOOST SURVIVAL AFT
# ==========================================================

def prepare_aft_labels(y):

    """
    Construit les bornes inférieure et supérieure
    requises par XGBoost Survival AFT.
    """

    # --------------------------------------------------
    # Vérification des données
    # --------------------------------------------------

    assert set(y["event_irt"].unique()) <= {0, 1}, (
        "event_irt doit contenir uniquement 0 (censuré) et 1 (événement)."
    )

    assert (y["time_to_event"] > 0).all(), (
        "time_to_event doit être strictement positif."
    )

    # --------------------------------------------------
    # Temps observé
    # --------------------------------------------------

    lower_bound = (
        y["time_to_event"]
        .to_numpy(dtype=float)
    )

    upper_bound = (
        lower_bound.copy()
    )

    # --------------------------------------------------
    # Patients censurés
    # --------------------------------------------------

    censored = (
        y["event_irt"] == 0
    )

    upper_bound[censored] = np.inf

    return (
        lower_bound,
        upper_bound
    )

In [ ]:
# ==========================================================
# CONSTRUCTION D'UNE DMATRIX XGBOOST SURVIVAL AFT
# ==========================================================

def build_dmatrix(
    X,
    lower_bound,
    upper_bound
):
    """
    Construit une DMatrix XGBoost contenant les bornes
    inférieure et supérieure requises par le modèle
    Survival AFT.

    Paramètres
    ----------
    X : pandas.DataFrame ou np.ndarray
        Variables explicatives.

    lower_bound : np.ndarray
        Temps observé (borne inférieure).

    upper_bound : np.ndarray
        Temps observé ou +∞ pour les patients censurés.

    Returns
    -------
    xgb.DMatrix
    """

    # --------------------------------------------------
    # Vérifications
    # --------------------------------------------------

    assert (
        len(X)
        == len(lower_bound)
        == len(upper_bound)
    ), "Dimensions incompatibles."

    assert np.all(
        lower_bound <= upper_bound
    ), "Certaines bornes inférieures sont supérieures aux bornes supérieures."

    # --------------------------------------------------
    # Construction de la DMatrix
    # --------------------------------------------------

    dmatrix = xgb.DMatrix(
        X,
        enable_categorical=True
    )

    # --------------------------------------------------
    # Labels Survival AFT
    # --------------------------------------------------

    dmatrix.set_float_info(
        "label_lower_bound",
        lower_bound
    )

    dmatrix.set_float_info(
        "label_upper_bound",
        upper_bound
    )

    return dmatrix

### FONCTION D'EVALUATION

In [ ]:
# ==========================================================
# ÉVALUATION DES PRÉDICTIONS
# ==========================================================

def evaluate_predictions(

    test_info,

    pred_time,

    k

):

    # ------------------------------------------------------
    # DataFrame des prédictions
    # ------------------------------------------------------

    df_pred = pd.DataFrame({

        "patient_id":

            test_info["patient_id"],

        "n_measures":

            test_info["n_measures"],

        "time_observed":

            test_info["time_to_event"],

        "event_irt":

            test_info["event_irt"],

        "pred_time":

            pred_time

    })

    # ------------------------------------------------------
    # Patients ayant réellement atteint k visites
    # ------------------------------------------------------

    df_pred = df_pred.loc[

        df_pred["n_measures"] >= k

    ].copy()

    # ------------------------------------------------------
    # Patients avec une prédiction valide
    # ------------------------------------------------------

    df_valid = df_pred.dropna(

        subset=["pred_time"]

    ).copy()

    # ------------------------------------------------------
    # C-index
    # ------------------------------------------------------

    if len(df_valid) > 0:

        cindex = concordance_index_censored(

            event_indicator=df_valid["event_irt"].astype(bool),

            event_time=df_valid["time_observed"],

            estimate=-df_valid["pred_time"]

        )[0]

    else:

        cindex = np.nan

    # ------------------------------------------------------
    # Patients avec événement
    # ------------------------------------------------------

    df_event = df_valid.loc[

        df_valid["event_irt"] == 1

    ].copy()

    # ------------------------------------------------------
    # MAE / RMSE
    # ------------------------------------------------------

    if len(df_event) > 0:

        mae = mean_absolute_error(

            df_event["time_observed"],

            df_event["pred_time"]

        )

        rmse = np.sqrt(

            mean_squared_error(

                df_event["time_observed"],

                df_event["pred_time"]

            )

        )

    else:

        mae = np.nan

        rmse = np.nan

    # ------------------------------------------------------
    # Résultats
    # ------------------------------------------------------

    return {

        "predictions":

            df_event,

        "cindex":

            cindex,

        "mae":

            mae,

        "rmse":

            rmse,

        "n_patients":

            len(df_pred),

        "n_events":

            int(

                df_pred["event_irt"].sum()

            ),

        "n_predicted":

            len(df_event)

    }

#### PARAMETRES DU MODELE

In [ ]:
# ==========================================================
# XGBOOST SURVIVAL AFT
# ==========================================================

import xgboost as xgb

from sksurv.metrics import concordance_index_censored


# ==========================================================
# PARAMÈTRES FIXES
# ==========================================================

FIXED_XGB_PARAMS = {

    "objective": "survival:aft",

    "eval_metric": "aft-nloglik",

    "tree_method": "hist",

    "seed": SEED

}


# ==========================================================
# PARAMÈTRES DE DÉPART
# ==========================================================

INITIAL_XGB_PARAMS = {

    **FIXED_XGB_PARAMS,

    "aft_loss_distribution": "normal",

    "aft_loss_distribution_scale": 0.25,

    "learning_rate": 0.03,

    "max_depth": 3,

    "min_child_weight": 1,

    "subsample": 0.8,

    "colsample_bytree": 0.8,

    "lambda": 1.0,

    "alpha": 0.0

}

#### CALIBRATION DES HYPERPARAMETRES

In [ ]:
# ==========================================================
# VALIDATION CROISÉE
# ==========================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error


def evaluate_cv(

    dataset,

    features,

    xgb_params,

    metric="cindex",

    n_splits=5,

    num_boost_round=200

):

    # ------------------------------------------------------
    # Préparation des données
    # ------------------------------------------------------

    (
        X,
        _,
        y,
        _,
        _
    ) = prepare_dataset(

        train_df=dataset,

        test_df=dataset,

        features=features

    )

    # ------------------------------------------------------
    # Validation croisée
    # ------------------------------------------------------

    cv = StratifiedKFold(

        n_splits=n_splits,

        shuffle=True,

        random_state=SEED

    )

    scores = []

    # ------------------------------------------------------
    # Boucle sur les folds
    # ------------------------------------------------------

    for train_idx, valid_idx in cv.split(

        X,

        y["event_irt"]

    ):

        # --------------------------------------------------
        # Split
        # --------------------------------------------------

        X_train = X.iloc[train_idx].copy()

        X_valid = X.iloc[valid_idx].copy()

        y_train = y.iloc[train_idx].copy()

        y_valid = y.iloc[valid_idx].copy()

        # --------------------------------------------------
        # Labels AFT
        # --------------------------------------------------

        train_lower, train_upper = prepare_aft_labels(
            y_train
        )

        valid_lower, valid_upper = prepare_aft_labels(
            y_valid
        )

        # --------------------------------------------------
        # DMatrix
        # --------------------------------------------------

        dtrain = build_dmatrix(

            X_train,

            train_lower,

            train_upper

        )

        dvalid = build_dmatrix(

            X_valid,

            valid_lower,

            valid_upper

        )

        # --------------------------------------------------
        # Entraînement
        # --------------------------------------------------

        booster = xgb.train(

            params=xgb_params,

            dtrain=dtrain,

            num_boost_round=num_boost_round,

            verbose_eval=False

        )

        # --------------------------------------------------
        # Prédictions
        # --------------------------------------------------

        pred = booster.predict(
            dvalid
        )

        # --------------------------------------------------
        # Vérification des prédictions
        # --------------------------------------------------

        if not np.all(np.isfinite(pred)):

            raise optuna.TrialPruned()

            
        pred = np.asarray(pred, dtype=np.float64)

        # --------------------------------------------------
        # C-index
        # --------------------------------------------------

        if metric == "cindex":

            score = concordance_index_censored(

                event_indicator=y_valid["event_irt"].astype(bool),

                event_time=y_valid["time_to_event"],

                estimate=-pred

            )[0]


        # --------------------------------------------------
        # MAE
        # --------------------------------------------------

        elif metric == "mae":

            event_mask = (

                y_valid["event_irt"] == 1

            )

            if event_mask.sum() == 0:

                continue

            score = mean_absolute_error(

                y_valid.loc[
                    event_mask,
                    "time_to_event"
                ],

                pred[event_mask]

            )

        else:

            raise ValueError(

                "metric doit être 'cindex' ou 'mae'."

            )

        scores.append(score)

    # ------------------------------------------------------
    # Score moyen
    # ------------------------------------------------------

    if metric == "cindex":

        return float(np.mean(scores))

    else:

        return float(np.mean(scores))

In [ ]:
# ==========================================================
# FONCTION OBJECTIF OPTUNA
# ==========================================================

import optuna


def objective(

    trial,

    dataset,

    features,

    metric="cindex"

):

    # ------------------------------------------------------
    # Hyperparamètres à optimiser
    # ------------------------------------------------------

    params = {

        **FIXED_XGB_PARAMS,

        "learning_rate": trial.suggest_float(

            "learning_rate",

            0.01,

            0.15,

            log=True

        ),

        "max_depth": trial.suggest_int(

            "max_depth",

            2,

            6

        ),

        "min_child_weight": trial.suggest_int(

            "min_child_weight",

            1,

            10

        ),

        "subsample": trial.suggest_float(

            "subsample",

            0.5,

            1.0

        ),

        "colsample_bytree": trial.suggest_float(

            "colsample_bytree",

            0.5,

            1.0

        ),

        "lambda": trial.suggest_float(

            "lambda",

            1e-3,

            10,

            log=True

        ),

        "alpha": trial.suggest_float(

            "alpha",

            1e-3,

            10,

            log=True

        ),

        "aft_loss_distribution": trial.suggest_categorical(

            "aft_loss_distribution",

            [

                "normal",

                "logistic",

                "extreme"

            ]

        ),

        "aft_loss_distribution_scale": trial.suggest_float(

            "aft_loss_distribution_scale",

            0.25,

            2.0,

            log=True

        )

    }

    # ------------------------------------------------------
    # Validation croisée
    # ------------------------------------------------------

    score = evaluate_cv(

        dataset=dataset,

        features=features,

        xgb_params=params,

        metric=metric

    )

    # ------------------------------------------------------
    # Score Optuna
    # ------------------------------------------------------

    if metric == "cindex":

        return score

    else:

        return score

In [ ]:
# ==========================================================
# OPTIMISATION DES HYPERPARAMÈTRES
# ==========================================================

from optuna.samplers import TPESampler


def optimize_xgb_hyperparameters(

    dataset,

    features,

    metric="cindex",

    n_trials=50,

    timeout=None

):

    # ------------------------------------------------------
    # Sens de l'optimisation
    # ------------------------------------------------------

    direction = (

        "maximize"

        if metric == "cindex"

        else "minimize"

    )

    # ------------------------------------------------------
    # Étude Optuna
    # ------------------------------------------------------

    study = optuna.create_study(

        direction=direction,

        sampler=TPESampler(

            seed=SEED

        )

    )

    # ------------------------------------------------------
    # Optimisation
    # ------------------------------------------------------

    study.optimize(

        lambda trial: objective(

            trial=trial,

            dataset=dataset,

            features=features,

            metric=metric

        ),

        n_trials=n_trials,

        timeout=timeout,

        show_progress_bar=True

    )

    # ------------------------------------------------------
    # Meilleurs paramètres
    # ------------------------------------------------------

    best_params = {

        **FIXED_XGB_PARAMS,

        **study.best_params

    }

    # ------------------------------------------------------
    # Résultats
    # ------------------------------------------------------

    print("\n==============================")

    print("OPTUNA TERMINÉ")

    print("==============================")

    print(f"Métrique : {metric}")

    print(f"Meilleur score : {study.best_value:.5f}")

    print("\nMeilleurs paramètres :\n")

    for key, value in best_params.items():

        print(f"{key:<30} : {value}")

    # ------------------------------------------------------
    # Sauvegarde
    # ------------------------------------------------------

    import os
    import json
    import joblib

    os.makedirs(

        "params",

        exist_ok=True

    )

    with open(

        f"params/best_xgb_params_{metric}.json",

        "w",

        encoding="utf-8"

    ) as f:

        json.dump(

            best_params,

            f,

            indent=4

        )

    joblib.dump(

        study,

        f"params/optuna_study_{metric}.pkl"

    )

    # ------------------------------------------------------
    # Retour
    # ------------------------------------------------------

    return (

        best_params,

        study

    )

In [ ]:
# ======================================
# OPTIMISATION 
# ======================================

BEST_XGB_PARAMS, study = optimize_xgb_hyperparameters(

    dataset=train_datasets[12],

    features=FEATURES,

    metric="mae",

    n_trials=50

)

In [ ]:
Résultats

↓

Influence du nombre de visites

↓

SHAP

↓

Analyse des erreurs

↓

Pourquoi certains patients sont difficiles ?

↓

Exemples individuels

↓

Discussion

#### FONCTION D'ENTRAINEMENT

In [ ]:
# ==========================================================
# XGBOOST SURVIVAL AFT
# ==========================================================

def run_xgb_survival(

    train_dataset,

    test_dataset,

    features,

    k,

    xgb_params=None,

    num_boost_round=200,

    model_name="LONGITUDINAL",

    verbose=True

):

    # ------------------------------------------------------
    # Paramètres
    # ------------------------------------------------------

    if xgb_params is None:

        xgb_params = BEST_XGB_PARAMS

    # ------------------------------------------------------
    # Préparation des données
    # ------------------------------------------------------

    (
        X_train,
        X_test,
        y_train_df,
        y_test_df,
        test_info

    ) = prepare_dataset(

        train_dataset,

        test_dataset,

        features

    )

    # ------------------------------------------------------
    # Vérification des dimensions
    # ------------------------------------------------------

    assert len(X_train) == len(y_train_df)

    assert len(X_test) == len(y_test_df)

    # ------------------------------------------------------
    # Labels AFT
    # ------------------------------------------------------

    train_lower, train_upper = prepare_aft_labels(
        y_train_df
    )

    test_lower, test_upper = prepare_aft_labels(
        y_test_df
    )

    # ------------------------------------------------------
    # DMatrix
    # ------------------------------------------------------

    dtrain = build_dmatrix(

        X_train,

        train_lower,

        train_upper

    )

    dtest = build_dmatrix(

        X_test,

        test_lower,

        test_upper

    )

    # ------------------------------------------------------
    # Entraînement
    # ------------------------------------------------------

    booster = xgb.train(

        params=xgb_params,

        dtrain=dtrain,

        num_boost_round=num_boost_round,

        evals=[

            (dtrain, "train"),

            (dtest, "test")

        ],

        verbose_eval=False

    )

    # ------------------------------------------------------
    # Prédictions
    # ------------------------------------------------------

    pred_train = booster.predict(
        dtrain
    )

    pred_test = booster.predict(
        dtest
    )

    # ------------------------------------------------------
    # C-index train
    # ------------------------------------------------------

    cindex_train = concordance_index_censored(

        event_indicator=y_train_df["event_irt"].astype(bool),

        event_time=y_train_df["time_to_event"],

        estimate=-pred_train

    )[0]

    # ------------------------------------------------------
    # Évaluation
    # ------------------------------------------------------

    evaluation = evaluate_predictions(

        test_info=test_info,

        pred_time=pred_test,

        k=k

    )

    # ------------------------------------------------------
    # Affichage
    # ------------------------------------------------------

    if verbose:

        print("=" * 60)

        print(f"XGB Survival AFT - {model_name}")

        print("=" * 60)

        print(
            f"C-index train : {cindex_train:.4f}"
        )

        print(
            f"C-index test  : {evaluation['cindex']:.4f}"
        )

        print(
            f"Patients test : {evaluation['n_patients']}"
        )

        print(
            f"Événements IRT : {evaluation['n_events']}"
        )

        print(
            f"Patients évalués : {evaluation['n_predicted']}"
        )

        print(
            f"MAE : {evaluation['mae']:.2f} années"
        )

        print(
            f"RMSE : {evaluation['rmse']:.2f} années"
        )

    # ------------------------------------------------------
    # Résultats
    # ------------------------------------------------------

    return {

        "model":

            booster,

        "X_test": X_test,

        "test_info": test_info,

        "event_predictions":

            evaluation["predictions"],

        "features":

            features,

        "xgb_params":

            xgb_params,

        "num_boost_round":

            num_boost_round,

        "cindex_train":

            cindex_train,

        "cindex_test":

            evaluation["cindex"],

        "mae":

            evaluation["mae"],

        "rmse":

            evaluation["rmse"],

        "n_patients":

            evaluation["n_patients"],

        "n_events":

            evaluation["n_events"],

        "n_predicted":

            evaluation["n_predicted"]

    }

In [ ]:
# ==========================================================
# ÉVALUATION DES MODÈLES XGBOOST SURVIVAL AFT
# ==========================================================

results_xgb = []

predictions_xgb = {}

models_xgb = {}

X_test_xgb = {}

test_info_xgb = {}

for k in range(1, 13):

    res = run_xgb_survival(

        train_dataset=train_datasets[k],

        test_dataset=internal_test_datasets[k],

        features=FEATURES,

        k=k,

        verbose=False

    )

    models_xgb[k] = res["model"]

    X_test_xgb[k] = res["X_test"]

    test_info_xgb[k] = res["test_info"]

    predictions_xgb[k] = res["event_predictions"]

    results_xgb.append({

        "model": "XGB",

        "n_visits": k,

        "n_patients": res["n_patients"],

        "n_events": res["n_events"],

        "n_predicted": res["n_predicted"],

        "cindex_train": res["cindex_train"],

        "cindex_test": res["cindex_test"],

        "mae": res["mae"],

        "rmse": res["rmse"]

    })

results_xgb = pd.DataFrame(results_xgb)

In [ ]:
display(
    results_xgb.round(3)
)

In [ ]:
predictions_xgb[1].head(20)

In [ ]:
pred = predictions_xgb[8].copy()

pred["error"] = (
    pred["pred_time"]
    - pred["time_observed"]
)

pred["absolute_error"] = pred["error"].abs()

display(
    pred[
        [
            "patient_id",
            "time_observed",
            "n_measures",
            "pred_time",
            "error",
            "absolute_error"
        ]
    ].round(2)
)

In [ ]:
pred["absolute_error"].describe()

In [ ]:
import plotly.express as px

fig = px.scatter(

    results_xgb,

    x="n_visits",

    y="mae",

    size="n_patients",

    color="cindex_test",

    hover_data={

        "n_patients": True,

        "n_events": True,

        "n_predicted": True,

        "cindex_test": ":.3f",

        "mae": ":.2f",

        "rmse": ":.2f"

    },

    labels={

        "n_visits": "Nombre de visites",

        "mae": "MAE (années)",

        "cindex_test": "C-index"

    },

    title="Précision du modèle selon le nombre de visites"

)

fig.update_traces(

    mode="markers+lines"

)

fig.update_layout(

    template="plotly_white",

    xaxis=dict(dtick=1)

)

fig.show()

In [ ]:
# output_dir = Path(
#     r".\tables\survival_models"
# )


#### ÉVALUATION DES PERFORMANCES SELON LE NOMBRE DE VISITES

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# ------------------------------------------------------
# MAE
# ------------------------------------------------------

fig.add_trace(

    go.Scatter(

        x=results_xgb["n_visits"],

        y=results_xgb["mae"],

        mode="lines+markers",

        name="MAE",

        yaxis="y1"

    )

)

# ------------------------------------------------------
# RMSE
# ------------------------------------------------------

fig.add_trace(

    go.Scatter(

        x=results_xgb["n_visits"],

        y=results_xgb["rmse"],

        mode="lines+markers",

        name="RMSE",

        yaxis="y1"

    )

)

# ------------------------------------------------------
# C-index
# ------------------------------------------------------

fig.add_trace(

    go.Scatter(

        x=results_xgb["n_visits"],

        y=results_xgb["cindex_test"],

        mode="lines+markers",

        name="C-index",

        yaxis="y2"

    )

)

# ------------------------------------------------------
# Layout
# ------------------------------------------------------

fig.update_layout(

    template="plotly_white",

    title="Évolution des performances du modèle selon le nombre de visites",

    xaxis_title="Nombre minimal de visites",

    yaxis=dict(

        title="Erreur (années)"

    ),

    yaxis2=dict(

        title="C-index",

        overlaying="y",

        side="right",

        range=[0.75, 1]

    ),

    hovermode="x unified",

    width=900,

    height=500

)

fig.show()

#### COMPROMIS ENTRE PERFORMANCES PRÉDICTIVES ET TAILLE DE LA COHORTE

In [ ]:
compromise = results_xgb.copy()

compromise["patients_retained"] = (

    100
    * compromise["n_patients"]
    / compromise.loc[0, "n_patients"]

)

compromise["events_retained"] = (

    100
    * compromise["n_events"]
    / compromise.loc[0, "n_events"]

)

display(

    compromise[

        [

            "n_visits",

            "n_patients",

            "patients_retained",

            "n_events",

            "events_retained",

            "mae",

            "rmse",

            "cindex_test"

        ]

    ].round(2)

)

**Interprétation :**

  Les performances s'améliorent nettement jusqu'à quatre visites, tout en conservant une proportion importante de la cohorte. L'ajout de visites supplémentaires permet encore de réduire le MAE, notamment jusqu'à six visites, mais au prix d'une diminution importante du nombre de patients et d'événements disponibles. Ainsi, selon que l'on privilégie la représentativité de la cohorte ou la précision des prédictions, un compromis raisonnable peut être situé entre quatre et six visites.
  

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# ------------------------------------------------------
# MAE
# ------------------------------------------------------

fig.add_trace(

    go.Scatter(

        x=compromise["n_visits"],

        y=compromise["mae"],

        mode="lines+markers",

        name="MAE",

        yaxis="y1"

    )

)

# ------------------------------------------------------
# Patients conservés
# ------------------------------------------------------

fig.add_trace(

    go.Scatter(

        x=compromise["n_visits"],

        y=compromise["patients_retained"],

        mode="lines+markers",

        name="% patients conservés",

        yaxis="y2"

    )

)

# ------------------------------------------------------
# Layout
# ------------------------------------------------------

fig.update_layout(

    template="plotly_white",

    title="Compromis entre précision des prédictions et taille de la cohorte",

    xaxis_title="Nombre minimal de visites",

    yaxis=dict(

        title="MAE (années)"

    ),

    yaxis2=dict(

        title="% de patients conservés",

        overlaying="y",

        side="right",

        range=[0, 105]

    ),

    hovermode="x unified",

    width=900,

    height=500

)

fig.show()

#### INTERPRÉTABILITÉ DU MODÈLE (SHAP)

In [ ]:
# ==========================================================
# CALCUL DES VALEURS SHAP
# ==========================================================

import xgboost as xgb

# ------------------------------------------------------
# Modèle retenu
# ------------------------------------------------------

model = models_xgb[4]

X_test = X_test_xgb[4].copy()

# ------------------------------------------------------
# DMatrix
# ------------------------------------------------------

dtest = xgb.DMatrix(

    X_test,

    enable_categorical=True

)

# ------------------------------------------------------
# Contributions SHAP
# ------------------------------------------------------

shap_values = model.predict(

    dtest,

    pred_contribs=True

)

In [ ]:
# ==========================================================
# OBJET SHAP
# ==========================================================

import shap

shap_explanation = shap.Explanation(

    values=shap_values[:, :-1],

    base_values=shap_values[:, -1],

    data=X_test,

    feature_names=X_test.columns.tolist()

)

In [ ]:
shap_importance = (

    pd.DataFrame({

        "feature": X_test.columns,

        "importance": np.abs(

            shap_explanation.values

        ).mean(axis=0)

    })

    .sort_values(

        "importance",

        ascending=False

    )

    .reset_index(drop=True)

)

display(
    shap_importance
)

In [ ]:
import plotly.express as px

# ------------------------------------------------------
# Importance globale des variables : Top 10 variables
# ------------------------------------------------------

top_features = (

    shap_importance

    .head(10)

    .sort_values(

        "importance",

        ascending=True

    )

)

# ------------------------------------------------------
# Graphique
# ------------------------------------------------------

fig = px.bar(

    top_features,

    x="importance",

    y="feature",

    orientation="h",

    text="importance",

    labels={

        "importance": "Importance moyenne |SHAP|",

        "feature": ""

    },

    title="Variables les plus importantes pour la prédiction du temps avant l'IRT"

)

fig.update_traces(

    texttemplate="%{text:.3f}",

    textposition="outside"

)

fig.update_layout(

    template="plotly_white",

    height=550,

    width=900,

    showlegend=False

)

fig.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,7))

shap.plots.beeswarm(

    shap_explanation,

    max_display=10,

    show=False

)

plt.tight_layout()

plt.show()

**Interprétation :**

  Les valeurs SHAP confirment que le modèle s'appuie principalement sur les indicateurs de fonction rénale pour prédire le temps avant l'IRT. Le DFG actuel (current_ckd_epi) est la variable la plus influente, suivi du DFG moyen, du DFG précédent et de la pente d'évolution du DFG (egfr_slope). Les patients présentant des valeurs faibles de DFG (points bleus) ont des contributions SHAP négatives, conduisant à une prédiction d'IRT plus précoce, tandis que des valeurs élevées de DFG (points rouges) sont associées à une prédiction plus tardive. À l'inverse, les variables cliniques comme la génétique ou les autres caractéristiques jouent un rôle plus limité dans les prédictions du modèle.

#### SHAP DEPENDENCE PLOT

In [ ]:
# -------------------------------------------------
# Fonction
# -------------------------------------------------

import pandas as pd
import plotly.express as px


def plot_shap_dependence_ckd(

    feature,

    X,

    shap_explanation,

    test_info=None

):

    df = pd.DataFrame({

        "feature_value": X[feature],

        "shap": shap_explanation.values[

            :,

            X.columns.get_loc(feature)

        ]

    })

    # --------------------------------------------------
    # Information supplémentaire au survol
    # --------------------------------------------------

    if test_info is not None:

        if "time_to_event" in test_info.columns:

            df["time_to_event"] = test_info["time_to_event"].values

        if "event_irt" in test_info.columns:

            df["event_irt"] = test_info["event_irt"].values

    # --------------------------------------------------
    # Scatter
    # --------------------------------------------------

    fig = px.scatter(

        df,

        x="feature_value",

        y="shap",

        color="time_to_event" if "time_to_event" in df.columns else None,

        color_continuous_scale="Viridis",

        opacity=0.75,

        trendline="lowess",

        labels={

            "feature_value": "DFG (CKD-EPI)",

            "shap": "Contribution SHAP",

            "time_to_event": "Temps avant IRT (années)"

        },

        title="Influence du DFG actuel sur la prédiction"

    )

    # --------------------------------------------------
    # Stades MRC
    # --------------------------------------------------

    stages = [

        (0, 15, "G5"),

        (15, 30, "G4"),

        (30, 45, "G3b"),

        (45, 60, "G3a"),

        (60, df["feature_value"].max(), "G1-G2")

    ]

    colors = [

        "rgba(220,20,60,0.08)",

        "rgba(255,140,0,0.08)",

        "rgba(255,215,0,0.08)",

        "rgba(50,205,50,0.08)",

        "rgba(30,144,255,0.05)"

    ]

    for (xmin, xmax, label), color in zip(stages, colors):

        fig.add_vrect(

            x0=xmin,

            x1=xmax,

            fillcolor=color,

            line_width=0

        )

        fig.add_annotation(

            x=(xmin+xmax)/2,

            y=df["shap"].max(),

            text=label,

            showarrow=False,

            yshift=18,

            font=dict(size=11)

        )

    # --------------------------------------------------
    # Ligne horizontale
    # --------------------------------------------------

    fig.add_hline(

        y=0,

        line_dash="dash",

        line_color="black"

    )

    # --------------------------------------------------
    # Mise en forme
    # --------------------------------------------------

    fig.update_layout(

        template="plotly_white",

        height=600,

        width=950,

        title_x=0.5

    )

    fig.show()

In [ ]:
# CURRENT DFG

plot_shap_dependence_ckd(

    feature="current_ckd_epi",

    X=X_test,

    shap_explanation=shap_explanation,

    test_info=test_info_xgb[4]

)

In [ ]:
# ==========================================================
# ÉVOLUTION DU C-INDEX
# ==========================================================

# ==========================================================
# ÉVOLUTION DU MAE
# ==========================================================

# ==========================================================
# ÉVOLUTION DU RMSE
# ==========================================================

# ==========================================================
# STABILISATION DES PERFORMANCES
# ==========================================================

# ==========================================================
# IMPORTANCE DES VARIABLES (SHAP)
# ==========================================================

# ==========================================================
# ANALYSE DES ERREURS DE PRÉDICTION
# ==========================================================

# ==========================================================
# ANALYSE DE CAS INDIVIDUELS
# ==========================================================



#### ANALYSE DES ERREURS DE PREDICTION

In [ ]:
# ==========================================================
# ANALYSE DES ERREURS DE PRÉDICTION
# ==========================================================

patient_analysis = (

    predictions_xgb[4]

    .copy()

)

# ------------------------------------------------------
# Erreur absolue
# ------------------------------------------------------

patient_analysis["absolute_error"] = (

    patient_analysis["pred_time"]

    - patient_analysis["time_observed"]

).abs()

display(
    patient_analysis.head()
)

In [ ]:
# ------------------------------------------------------
# Quartiles
# ------------------------------------------------------

q1 = patient_analysis["absolute_error"].quantile(0.25)

q3 = patient_analysis["absolute_error"].quantile(0.75)

# ------------------------------------------------------
# Groupes
# ------------------------------------------------------

patient_analysis["prediction_group"] = "Intermediate"

patient_analysis.loc[

    patient_analysis["absolute_error"] <= q1,

    "prediction_group"

] = "Well predicted"

patient_analysis.loc[

    patient_analysis["absolute_error"] >= q3,

    "prediction_group"

] = "Poorly predicted"

In [ ]:
patient_analysis.groupby(

    "prediction_group"

).agg(

    n_patients=("patient_id", "count"),

    median_error=("absolute_error", "median"),

    mean_error=("absolute_error", "mean")

)

In [ ]:
import plotly.express as px

fig = px.histogram(

    patient_analysis,

    x="absolute_error",

    nbins=30,

    color="prediction_group",

    barmode="overlay",

    opacity=0.75,

    labels={

        "absolute_error": "Erreur absolue (années)",

        "prediction_group": ""

    },

    title="Distribution des erreurs de prédiction"

)

fig.update_layout(

    template="plotly_white"

)

fig.show()

In [ ]:
analysis_dataset = (

    internal_test_datasets[4]

    .to_pandas()

)

analysis_dataset = analysis_dataset.merge(

    patient_analysis[

        [

            "patient_id",

            "absolute_error",

            "prediction_group"

        ]

    ],

    on="patient_id",

    how="inner"

)

#### Les variables qui différent selon les groupes

In [ ]:
NUMERIC_FEATURES = [

    "age_creat",

    "current_ckd_epi",

    "baseline_ckd_epi",

    "previous_ckd_epi",

    "delta_ckd_from_baseline",

    "recent_annualized_change",

    "egfr_mean",

    "egfr_std",

    "egfr_slope",

    "time_since_last_measure",

    "followup_density",

    "n_measures",

    "years_since_hypertension",

    "years_since_urologic_event"

]

CATEGORICAL_FEATURES = [

    "sex",

    "genetic",

    "ever_hypertension",

    "ever_urologic_event"

]

In [ ]:
from scipy.stats import kruskal

numeric_results = []

for variable in NUMERIC_FEATURES:

    groups = [

        g[variable].dropna()

        for _, g in analysis_dataset.groupby(

            "prediction_group"

        )

    ]

    statistic, pvalue = kruskal(*groups)

    numeric_results.append({

        "variable": variable,

        "H": statistic,

        "p_value": pvalue

    })

numeric_results = (

    pd.DataFrame(numeric_results)

    .sort_values("p_value")

)

display(
    numeric_results
)

In [ ]:
from scipy.stats import chi2_contingency

categorical_results = []

for variable in CATEGORICAL_FEATURES:

    contingency = pd.crosstab(

        analysis_dataset[variable],

        analysis_dataset["prediction_group"]

    )

    chi2, pvalue, _, _ = chi2_contingency(

        contingency

    )

    categorical_results.append({

        "variable": variable,

        "chi2": chi2,

        "p_value": pvalue

    })

categorical_results = (

    pd.DataFrame(categorical_results)

    .sort_values("p_value")

)

display(
    categorical_results
)

In [ ]:
# ==========================================================
# TABLE D'ANALYSE DES PATIENTS AVEC ÉVÉNEMENT
# ==========================================================

analysis_table = (

    patient_analysis

    .merge(

        analysis_dataset,

        on="patient_id",

        how="left",

        suffixes=("", "_snapshot")

    )

)

analysis_table = analysis_table.drop(

    columns=[

        "n_measures_snapshot",

        "event_irt_snapshot",

        "absolute_error_snapshot",

        "prediction_group_snapshot"

    ],

    errors="ignore"

)

print(analysis_table.shape)

In [ ]:
# ==========================================================
# TABLEAU DES 5 MEILLEURS ET 5 PIRES PATIENTS
# ==========================================================

best = (

    analysis_table

    .nsmallest(

        5,

        "absolute_error"

    )

    .assign(group="Best")

)

worst = (

    analysis_table

    .nlargest(

        5,

        "absolute_error"

    )

    .assign(group="Worst")

)

summary = pd.concat(

    [

        best,

        worst

    ],

    ignore_index=True

)

summary = summary[[

    "group",

    "patient_id",

    "absolute_error",

    "time_observed",

    "pred_time",

    "current_ckd_epi",

    "baseline_ckd_epi",

    "egfr_slope",

    "recent_annualized_change",

    "egfr_std",

    "n_measures",

    "followup_density",

    "genetic",

    "sex"

]]

display(summary.round(2))

#### Construire une table de convergence

In [ ]:
# ==========================================================
# ÉVOLUTION DES PRÉDICTIONS
# ==========================================================

prediction_history = []

for k in range(1, 13):

    df = predictions_xgb[k].copy()

    df["n_visits"] = k

    prediction_history.append(df)

prediction_history = pd.concat(

    prediction_history,

    ignore_index=True

)
prediction_history.head()

In [ ]:
best_ids = (

    best["patient_id"]

    .tolist()

)

worst_ids = (

    worst["patient_id"]

    .tolist()

)

plot_df = prediction_history[

    prediction_history["patient_id"].isin(

        best_ids + worst_ids

    )

].copy()

plot_df["group"] = np.where(

    plot_df["patient_id"].isin(best_ids),

    "Best",

    "Worst"

)

In [ ]:
# ==========================================================
# HISTORIQUE DES PRÉDICTIONS
# ==========================================================

history = []

selected_ids = summary["patient_id"].tolist()

for k in range(1, 13):

    df = predictions_xgb[k].copy()

    df = df[

        df["patient_id"].isin(

            selected_ids

        )

    ].copy()

    df["n_visits"] = k

    df["absolute_error"] = (

        df["pred_time"]

        - df["time_observed"]

    ).abs()

    history.append(df)

history = pd.concat(

    history,

    ignore_index=True

)

# ----------------------------------------------------------
# Groupe (Best / Worst)
# ----------------------------------------------------------

history = history.merge(

    summary[

        [

            "patient_id",

            "group"

        ]

    ],

    on="patient_id",

    how="left"

)

# ----------------------------------------------------------
# Variables longitudinales
# ----------------------------------------------------------

dynamic = (

    internal_test_dynamic

    .select(

        [

            "patient_id",

            "n_measures",

            "current_ckd_epi",

            "age_creat",

            "creat_date"

        ]

    )

    .to_pandas()

)

history = history.merge(

    dynamic,

    on=[

        "patient_id",

        "n_measures"

    ],

    how="left"

)

# ----------------------------------------------------------
# Vérification
# ----------------------------------------------------------

display(history.head())

print(history.shape)

In [ ]:
# ==========================================================
# ÉVOLUTION DES PRÉDICTIONS PAR PATIENT
# ==========================================================

from plotly.subplots import make_subplots
import plotly.graph_objects as go

best_ids = (
    summary
    .query("group == 'Best'")
    ["patient_id"]
    .tolist()
)

worst_ids = (
    summary
    .query("group == 'Worst'")
    ["patient_id"]
    .tolist()
)

# ----------------------------------------------------------
# Titres
# ----------------------------------------------------------

titles = []

for patient in best_ids:

    row = summary.loc[
        summary["patient_id"] == patient
    ].iloc[0]

    titles.append(
        f"Patient {patient}<br>"
        f"Erreur = {row['absolute_error']:.2f} an"
    )

for patient in worst_ids:

    row = summary.loc[
        summary["patient_id"] == patient
    ].iloc[0]

    titles.append(
        f"Patient {patient}<br>"
        f"Erreur = {row['absolute_error']:.2f} ans"
    )

# ----------------------------------------------------------
# Figure
# ----------------------------------------------------------

fig = make_subplots(

    rows=5,

    cols=2,

    subplot_titles=titles,

    vertical_spacing=0.07,

    horizontal_spacing=0.10

)

# ==========================================================
# PATIENTS BEST
# ==========================================================

for i, patient in enumerate(best_ids):

    df = (
        history
        .query("patient_id == @patient")
        .sort_values("n_visits")
    )

    custom = df[[
        "patient_id",
        "creat_date",
        "age_creat",
        "current_ckd_epi",
        "time_observed",
        "pred_time",
        "absolute_error"
    ]]

    # Temps réel

    fig.add_trace(

        go.Scatter(

            x=df["n_visits"],

            y=df["time_observed"],

            mode="lines+markers",

            name="Temps réel",

            legendgroup="real",

            showlegend=(i == 0),

            line=dict(
                color="black",
                width=3
            ),

            marker=dict(
                size=8
            ),

            customdata=custom,

            hovertemplate=
            "<b>Patient %{customdata[0]}</b><br>"
            "Date : %{customdata[1]}<br>"
            "Age : %{customdata[2]:.1f} ans<br>"
            "Nombre de visites : %{x}<br>"
            "DFG : %{customdata[3]:.1f}<br>"
            "Temps réel : %{y:.2f} ans<br>"
            "Temps prédit : %{customdata[4]:.2f} ans<br>"
            "Erreur absolue : %{customdata[5]:.2f} ans"
            "<extra></extra>"

        ),

        row=i+1,

        col=1

    )

    # Temps prédit

    fig.add_trace(

        go.Scatter(

            x=df["n_visits"],

            y=df["pred_time"],

            mode="lines+markers",

            name="Temps prédit",

            legendgroup="pred",

            showlegend=(i == 0),

            line=dict(
                color="firebrick",
                width=2
            ),

            marker=dict(
                size=8
            ),

            customdata=custom,

            hovertemplate=
            "<b>Patient %{customdata[0]}</b><br>"
            "Date : %{customdata[1]}<br>"
            "Age : %{customdata[2]:.1f} ans<br>"
            "Nombre de visites : %{x}<br>"
            "DFG : %{customdata[3]:.1f}<br>"
            "Temps réel : %{y:.2f} ans<br>"
            "Temps prédit : %{customdata[4]:.2f} ans<br>"
            "Erreur absolue : %{customdata[5]:.2f} ans"
            "<extra></extra>"

        ),

        row=i+1,

        col=1

    )

# ==========================================================
# PATIENTS WORST
# ==========================================================

for i, patient in enumerate(worst_ids):

    df = (
        history
        .query("patient_id == @patient")
        .sort_values("n_visits")
    )

    custom = df[[
        "patient_id",
        "creat_date",
        "age_creat",
        "current_ckd_epi",
        "time_observed",
        "pred_time",
        "absolute_error"
    ]]

    # Temps réel

    fig.add_trace(

        go.Scatter(

            x=df["n_visits"],

            y=df["time_observed"],

            mode="lines+markers",

            legendgroup="real",

            showlegend=False,

            line=dict(
                color="black",
                width=3
            ),

            marker=dict(
                size=8
            ),

            customdata=custom,

            hovertemplate=
            "<b>Patient %{customdata[0]}</b><br>"
            "Date : %{customdata[1]}<br>"
            "Age : %{customdata[2]:.1f} ans<br>"
            "Nombre de visites : %{x}<br>"
            "DFG : %{customdata[3]:.1f}<br>"
            "Temps réel : %{y:.2f} ans<br>"
            "Temps prédit : %{customdata[4]:.2f} ans<br>"
            "Erreur absolue : %{customdata[5]:.2f} ans"
            "<extra></extra>"

        ),

        row=i+1,

        col=2

    )

    # Temps prédit

    fig.add_trace(

        go.Scatter(

            x=df["n_visits"],

            y=df["pred_time"],

            mode="lines+markers",

            legendgroup="pred",

            showlegend=False,

            line=dict(
                color="firebrick",
                width=2
            ),

            marker=dict(
                size=8
            ),

            customdata=custom,

            hovertemplate=
            "<b>Patient %{customdata[0]}</b><br>"
            "Date : %{customdata[1]}<br>"
            "Age : %{customdata[2]:.1f} ans<br>"
            "Nombre de visites : %{x}<br>"
            "DFG : %{customdata[3]:.1f}<br>"
            "Temps réel : %{y:.2f} ans<br>"
            "Temps prédit : %{customdata[4]:.2f} ans<br>"
            "Erreur absolue : %{customdata[5]:.2f} ans"
            "<extra></extra>"

        ),

        row=i+1,

        col=2

    )

# ==========================================================
# MISE EN FORME
# ==========================================================

fig.update_xaxes(

    title_text="Nombre de visites",

    dtick=1

)

fig.update_yaxes(

    title_text="Temps restant avant IRT (années)"

)

fig.update_layout(

    template="plotly_white",

    height=1500,

    width=1200,

    title=(

        "Évolution du temps restant réel et prédit selon le nombre de visites"

        "<br><sup>Comparaison entre les patients les mieux et les moins bien prédits</sup>"

    ),

    legend=dict(

        orientation="h",

        x=0.5,

        xanchor="center",

        y=1.03

    ),

    hoverlabel=dict(

        bgcolor="white",

        font_size=12

    )

)

fig.show()

In [ ]:
# ==========================================================
# TEMPS ENTRE DEUX VISITES
# ==========================================================

intervals = (

    internal_test_dynamic

    .select(

        "mean_measurement_interval"

    )

    .to_pandas()

)

display(

    intervals.describe().round(2)

)

median_interval = (

    intervals["mean_measurement_interval"]

    .median()

)

print(

    f"Temps médian entre deux visites : {median_interval:.2f} années"

)

print(

    f"Soit environ {median_interval * 12:.1f} mois"

)

In [ ]:
# ==========================================================
# DURÉE DE SUIVI ESTIMÉE
# ==========================================================

followup_summary = results_xgb.copy()

followup_summary["median_followup_years"] = (

    (followup_summary["n_visits"] - 1)

    * median_interval

)

followup_summary["median_followup_months"] = (

    followup_summary["median_followup_years"]

    * 12

)

display(

    followup_summary[[

        "n_visits",

        "median_followup_years",

        "median_followup_months",

        "mae",

        "cindex_test",

        "n_patients"

    ]]

    .round(2)

)

In [ ]:
import plotly.express as px

fig = px.line(

    followup_summary,

    x="median_followup_months",

    y="mae",

    markers=True,

    title=(

        "Évolution de l'erreur de prédiction "

        "en fonction du temps médian de suivi"

    ),

    labels={

        "median_followup_months":

            "Temps médian de suivi (mois)",

        "mae":

            "MAE (années)"

    }

)

fig.update_layout(

    template="plotly_white"

)

fig.show()

#### Prédiction des catégories

In [ ]:
FEATURES_SN = [
    
    "age_creat",
    "sex",
    "genetic",
    "current_ckd_epi",
    "baseline_ckd_epi",
    "previous_ckd_epi",
    "percent_decline_from_baseline",
    "n_measures",
    "time_since_first_measure",
    "followup_density",
    "hypertension_at_t",
    "urologic_event_at_t",

]

In [ ]:
# ==========================================================
# MATRICES DES VARIABLES EXPLICATIVES
# ==========================================================

X_train = (
    train_dynamic
    .select(FEATURES_SN)
    .to_pandas()
)

X_test = (
    internal_test_dynamic
    .select(FEATURES_SN)
    .to_pandas()
)

In [ ]:
# ==========================================================
# VARIABLES DE SURVIE
# ==========================================================

y_train = (
    train_dynamic
    .select([
        "time_to_event",
        "event_irt"
    ])
    .to_pandas()
)

y_test = (
    internal_test_dynamic
    .select([
        "time_to_event",
        "event_irt"
    ])
    .to_pandas()
)

In [ ]:
# ==========================================================
# ENCODAGE DES VARIABLES CATÉGORIELLES
# ==========================================================

categorical_features = [
    "sex",
    "genetic",
]

# Encodage du jeu d'entraînement
X_train = pd.get_dummies(
    X_train,
    columns=categorical_features,
    dtype=int,
)

# Encodage du jeu de test
X_test = pd.get_dummies(
    X_test,
    columns=categorical_features,
    dtype=int,
)

# Garantir les mêmes colonnes entre train et test
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0,
)

In [ ]:
# ==========================================================
# LABELS AFT
# ==========================================================

train_lower, train_upper = prepare_aft_labels(y_train)

test_lower, test_upper = prepare_aft_labels(y_test)

In [ ]:
# ==========================================================
# DMatrix
# ==========================================================

dtrain = build_dmatrix(
    X_train,
    train_lower,
    train_upper
)

dtest = build_dmatrix(
    X_test,
    test_lower,
    test_upper
)

In [ ]:
# ==========================================================
# ENTRAINEMENT
# ==========================================================

booster = xgb.train(
    params=BEST_XGB_PARAMS,
    dtrain=dtrain,
    num_boost_round=200,
    verbose_eval=False
)

In [ ]:
# ==========================================================
# PREDICTIONS
# ==========================================================

pred_train = booster.predict(dtrain)

pred_test = booster.predict(dtest)

In [ ]:
# ==========================================================
# C-INDEX
# ==========================================================

from sksurv.metrics import concordance_index_censored

cindex_train = concordance_index_censored(

    event_indicator=y_train["event_irt"].astype(bool),

    event_time=y_train["time_to_event"],

    estimate=-pred_train

)[0]

cindex_test = concordance_index_censored(

    event_indicator=y_test["event_irt"].astype(bool),

    event_time=y_test["time_to_event"],

    estimate=-pred_test

)[0]

print(f"C-index train : {cindex_train:.4f}")
print(f"C-index test  : {cindex_test:.4f}")

In [ ]:
def predicted_horizon(t):

    if t < 5:
        return "<5"
    elif t < 10:
        return "5-10"
    elif t < 15:
        return "10-15"
    else:
        return ">15"

preview["Horizon prédit"] = preview["pred_time"].apply(predicted_horizon)

In [ ]:
# ==========================================================
# PROBABILITES DES HORIZONS
# ==========================================================

p_less5 = 1 - survival_probability(5, pred_test)

p_5_10 = survival_probability(5, pred_test) - survival_probability(10, pred_test)

p_10_15 = survival_probability(10, pred_test) - survival_probability(15, pred_test)

p_more15 = survival_probability(15, pred_test)

In [ ]:
# ==========================================================
# TABLEAU DES PROBABILITES
# ==========================================================

import pandas as pd

predictions = pd.DataFrame({

    "<5": p_less5,

    "5-10": p_5_10,

    "10-15": p_10_15,

    ">15": p_more15

})

In [ ]:
# ==========================================================
# HORIZON PREDIT
# ==========================================================

predictions["Horizon prédit"] = predictions.idxmax(axis=1)

In [ ]:
# ==========================================================
# PREVIEW
# ==========================================================

preview = (
    internal_test_dynamic
    .to_pandas()
    .copy()
)

preview["pred_time"] = pred_test

preview["Horizon prédit"] = predictions["Horizon prédit"].values

In [ ]:
# ==========================================================
# HORIZON REEL
# ==========================================================

import numpy as np

def true_horizon(row):

    t = row["time_to_event"]
    event = row["event_irt"]

    if event:

        if t < 5:
            return "<5"

        elif t < 10:
            return "5-10"

        elif t < 15:
            return "10-15"

        else:
            return ">15"

    else:

        # on ne connaît pas encore le vrai horizon

        if t < 15:
            return np.nan

        else:
            return ">15"

In [ ]:
preview["Horizon réel"] = preview.apply(
    true_horizon,
    axis=1
)

In [ ]:
preview = preview.dropna(
    subset=["Horizon réel"]
).reset_index(drop=True)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(

    preview["Horizon réel"],

    preview["Horizon prédit"]

)

print(f"Accuracy : {accuracy:.3f}")

In [ ]:
print(pred_test[:10])

In [ ]:
print(preview[["pred_time", "time_to_event"]].head(10))

In [ ]:
# ==========================================================
# HORIZON PRÉDIT À PARTIR DU TEMPS PRÉDIT
# ==========================================================

def predicted_horizon(t):

    if t < 5:
        return "<5"

    elif t < 10:
        return "5-10"

    elif t < 15:
        return "10-15"

    else:
        return ">15"


preview = internal_test_dynamic.to_pandas().copy()

preview["pred_time"] = pred_test

preview["Horizon prédit"] = preview["pred_time"].apply(predicted_horizon)

In [ ]:
preview["Horizon réel"] = preview.apply(
    true_horizon,
    axis=1
)

preview = preview.dropna(
    subset=["Horizon réel"]
).reset_index(drop=True)

In [ ]:
from sklearn.metrics import confusion_matrix

labels = ["<5", "5-10", "10-15", ">15"]

cm = confusion_matrix(
    preview["Horizon réel"],
    preview["Horizon prédit"],
    labels=labels
)

print(cm)

In [ ]:
# ==========================================================
# ACCURACY
# ==========================================================

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    preview["Horizon réel"],
    preview["Horizon prédit"]
)

print(f"Accuracy : {accuracy:.3f}")

In [ ]:
# ==========================================================
# CLASSIFICATION REPORT
# ==========================================================

from sklearn.metrics import classification_report

labels = [
    "<5",
    "5-10",
    "10-15",
    ">15"
]

print(
    classification_report(
        preview["Horizon réel"],
        preview["Horizon prédit"],
        labels=labels,
        digits=3
    )
)

In [ ]:
# ==========================================================
# MATRICE DE CONFUSION
# ==========================================================

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    preview["Horizon réel"],
    preview["Horizon prédit"],
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

print(cm_df)

In [ ]:
# ==========================================================
# MATRICE NORMALISEE
# ==========================================================

cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]

cm_norm = pd.DataFrame(
    cm_norm,
    index=labels,
    columns=labels
)

print(cm_norm.round(3))

In [ ]:
# ==========================================================
# EVALUATION COMPLETE D'UN CLASSIFIEUR
# ==========================================================

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix
)


def evaluate_classifier(preview):

    y_true = preview["Horizon réel"]

    y_pred = preview["Horizon prédit"]

    print("=" * 70)
    print("PERFORMANCES")
    print("=" * 70)

    print(f"Accuracy            : {accuracy_score(y_true,y_pred):.3f}")

    print(f"Balanced Accuracy   : {balanced_accuracy_score(y_true,y_pred):.3f}")

    print(f"Precision macro     : {precision_score(y_true,y_pred,average='macro'):.3f}")

    print(f"Recall macro        : {recall_score(y_true,y_pred,average='macro'):.3f}")

    print(f"F1 macro            : {f1_score(y_true,y_pred,average='macro'):.3f}")

    print(f"Cohen Kappa         : {cohen_kappa_score(y_true,y_pred):.3f}")

    print(f"MCC                 : {matthews_corrcoef(y_true,y_pred):.3f}")

    print()

    print("=" * 70)
    print("CLASSIFICATION REPORT")
    print("=" * 70)

    print(classification_report(y_true,y_pred,digits=3))

    labels=["<5","5-10","10-15",">15"]

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    cm = pd.DataFrame(
        cm,
        index=labels,
        columns=labels
    )

    return cm

cm_xgb = evaluate_classifier(preview)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,5))

sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1
)

plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.title("Matrice de confusion normalisée - RSF")

plt.tight_layout()
plt.show()

In [ ]:
(
    train_dynamic
    .group_by("patient_id")
    .len()
    .rename({"len": "n_visites"})
    .group_by("n_visites")
    .len()
    .rename({"len": "n_patients"})
    .sort("n_visites")
)
